# GPU EDA Parquet Visualizations - Local Kubeflow Edition

This notebook is the local visualization layer for the 3-notebook workflow.

## Inputs

A Parquet export directory containing:

- `raw_sample`
- `corr_summary`
- `corr_over_time`
- `density_bins`
- `density_over_time`
- `bin_metadata`
- `trend_summary`
- `manifest.json`

## Outputs

When you run this notebook it will:

1. Render the requested visuals inline in the notebook.
2. Save a PNG summary pack to disk.
3. Build a single HTML report file that embeds the saved charts and summary tables.
4. Zip the PNG pack for easy download/share.

## New Features

- Segmented visualizations by: gen, region_rollup, customer_segment, product_segment, is_training
- Support for both p50 and p95 correlation views
- IQR-scaled drift score heatmap
- Trailing 15-month window for monthly views
- Exclusion of large-scale metrics from boxplots

In [1]:

# ========================================
# PACKAGE INSTALLATION
# ========================================
import importlib
import importlib.metadata as importlib_metadata
import subprocess
import sys
import site


def is_module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False


def install_if_missing(pip_name: str, import_name: str = None):
    import_name = import_name or pip_name
    dist_name = pip_name.split('==', 1)[0]
    required_version = pip_name.split('==', 1)[1] if '==' in pip_name else None

    if not is_module_available(import_name):
        print(f'Installing {pip_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', pip_name])
        return

    if required_version:
        try:
            installed_version = importlib_metadata.version(dist_name)
        except importlib_metadata.PackageNotFoundError:
            installed_version = None
        if installed_version != required_version:
            print(f'Upgrading {dist_name} from {installed_version or "unknown"} to {required_version} ...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '--upgrade', pip_name])
            return

    print(f'✓ {pip_name} already installed')


print('=' * 60)
print('INSTALLING LOCAL VISUALIZATION DEPENDENCIES')
print('=' * 60)

install_if_missing('numpy', 'numpy')
install_if_missing('pandas', 'pandas')
install_if_missing('pyarrow', 'pyarrow')
install_if_missing('matplotlib', 'matplotlib')
install_if_missing('seaborn', 'seaborn')
install_if_missing('scipy', 'scipy')
install_if_missing('jinja2', 'jinja2')

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

print('✓ Dependency validation complete')
print('=' * 60)


INSTALLING LOCAL VISUALIZATION DEPENDENCIES
✓ numpy already installed
✓ pandas already installed
✓ pyarrow already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ scipy already installed
✓ jinja2 already installed
✓ Dependency validation complete


In [2]:
# ========================================
# CONFIGURATION
# ========================================
from pathlib import Path
from datetime import datetime

print('=' * 60)
print('RUNTIME CONFIGURATION')
print('=' * 60)

PARQUET_ROOT = Path('/home/coreweave/dcgm_eda_parquet')
EXPORT_TAG = None  # set to a specific export folder name, or leave None to auto-pick the latest
OUTPUT_DIR = Path('/home/coreweave/dcgm_eda_viz_outputs')
FIG_DIR = OUTPUT_DIR / 'figures'
PNG_ZIP_PATH = OUTPUT_DIR / 'dcgm_eda_png_summary_pack.zip'
HTML_REPORT_PATH = OUTPUT_DIR / 'dcgm_eda_visual_summary.html'

# Split report configuration
SPLIT_HTML_REPORTS = True  # Set to True to create separate HTML files per segment
HTML_REPORT_DIR = OUTPUT_DIR / 'html_reports'  # Directory for split reports
HTML_REPORT_DIR.mkdir(parents=True, exist_ok=True) if SPLIT_HTML_REPORTS else None

# Updated configuration parameters
MAX_PAIRPLOT_ROWS = 4000
MAX_SCATTER_ROWS_PER_MONTH = 2500
MAX_MONTHS = 15  # Limit to trailing 15 months for monthly views
DEFAULT_CORR_METHOD = 'pearson'
CORR_GROUPS_TO_RENDER = ['p50', 'p95']  # Support both p50 and p95 correlation views
DPI = 140
SEED = 42

# Metrics to exclude from all-metric box-and-whisker plots
BOXPLOT_EXCLUDE_METRICS = ['vram_usage', 'redfish_power', 'chip_power', 'sm_clock', 'tensor_tflops','tensor_tflops_sm']

# Segmentation dimensions to render
SEGMENT_DIMS_TO_RENDER = ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
MAX_SEGMENT_VALUES_PER_DIM = 10  # Limit segment values to avoid chart explosion
MIN_SEGMENT_ROWS = 100  # Minimum rows to render a segment

# Feature flags
RENDER_GLOBAL_VIEWS = True  # Keep unsegmented global views
RENDER_SEGMENTED_VIEWS = True  # Render segmented views

CORE_METRICS = ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm','chip_power', 'redfish_power', 'sm_active']
EXTENDED_METRICS = ['dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy']
INDEX_METRICS = [
    'compute_occupancy_index',
    'memory_boundness_index',
    'memory_intensity_index',
    'power_pct_max_index',
    'compute_pct_max_index',
    'TFLOPS_per_watt_efficiency',
]
EFFICIENCY_INDEX_METRICS = [
    'TFLOPS_per_watt_efficiency',
    'power_pct_max_index',
    'compute_pct_max_index',
]

WORKLOAD_INDEX_METRICS = [
    'compute_occupancy_index',
    'memory_boundness_index',
    'memory_intensity_index',
]

DIAGNOSTIC_INDEX_METRICS = [
    'tensor_util_dram_index',
    'compute_to_memory_ratio_index',
    'tensor_to_gpu_ratio_index',
]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'PARQUET_ROOT: {PARQUET_ROOT}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')
print(f'HTML report : {HTML_REPORT_PATH}')
print(f'PNG zip     : {PNG_ZIP_PATH}')
print(f'MAX_MONTHS  : {MAX_MONTHS}')
print(f'CORR_GROUPS : {CORR_GROUPS_TO_RENDER}')
print(f'BOXPLOT_EXCLUDE: {BOXPLOT_EXCLUDE_METRICS}')
print('=' * 60)


RUNTIME CONFIGURATION
PARQUET_ROOT: /home/coreweave/dcgm_eda_parquet
OUTPUT_DIR  : /home/coreweave/dcgm_eda_viz_outputs
HTML report : /home/coreweave/dcgm_eda_viz_outputs/dcgm_eda_visual_summary.html
PNG zip     : /home/coreweave/dcgm_eda_viz_outputs/dcgm_eda_png_summary_pack.zip
MAX_MONTHS  : 15
CORR_GROUPS : ['p50', 'p95']
BOXPLOT_EXCLUDE: ['vram_usage', 'redfish_power', 'chip_power', 'sm_clock', 'tensor_tflops', 'tensor_tflops_sm']


In [3]:
# ========================================
# IMPORTS + HELPERS
# ========================================
import json
import math
import zipfile
from itertools import combinations
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from jinja2 import Template
import base64
import pyarrow.dataset as ds
import pyarrow.compute as pc


sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = DPI
plt.rcParams['savefig.dpi'] = DPI

FIGURES = []
REPORT_TABLES = []
REPORT_SEGMENTS = {}  # Track segments for HTML report organization


def discover_export_root(base_root: Path, export_tag=None) -> Path:
    if export_tag:
        candidate = base_root / export_tag
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f'Export tag not found: {candidate}')
    if (base_root / 'manifest.json').exists():
        return base_root
    candidates = sorted([p for p in base_root.iterdir() if p.is_dir()], reverse=True)
    for c in candidates:
        if (c / 'manifest.json').exists():
            return c
    raise FileNotFoundError(f'Could not find manifest.json under {base_root}')


def optimize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif df[col].dtype == "object":
            nunique = df[col].nunique(dropna=False)
            if nunique > 0 and nunique < max(1000, len(df) * 0.2):
                df[col] = df[col].astype("category")
    return df


def discover_available_months(root: Path, dataset_name: str, month_col_candidates: List[str]) -> List[str]:
    path = root / dataset_name
    if not path.exists():
        return []

    dataset = ds.dataset(path, format="parquet")
    cols = set(dataset.schema.names)

    month_col = None
    for c in month_col_candidates:
        if c in cols:
            month_col = c
            break

    if month_col is None:
        return []

    table = dataset.to_table(columns=[month_col])
    pdf = table.to_pandas()
    months = monthify(pdf[month_col])
    return select_months(months, max_months=None)


def load_parquet_dataset(
    root: Path,
    name: str,
    columns: Optional[List[str]] = None,
    month_filter: Optional[List[str]] = None,
    month_col: Optional[str] = None,
) -> pd.DataFrame:
    path = root / name
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset path: {path}")

    dataset = ds.dataset(path, format="parquet")

    use_columns = columns
    if use_columns is not None:
        available = set(dataset.schema.names)
        use_columns = [c for c in use_columns if c in available]

    table = dataset.to_table(columns=use_columns)
    pdf = table.to_pandas()

    if month_filter and month_col and month_col in pdf.columns:
        if month_col != "month_period":
            pdf["month_period"] = monthify(pdf[month_col])
        else:
            pdf["month_period"] = monthify(pdf["month_period"])
        pdf = pdf[pdf["month_period"].isin(month_filter)].copy()

    return optimize_dataframe(pdf)


def save_figure(fig, filename: str, title: str, kind: str, segment_context: str = 'global', close: bool = True):
    out_path = FIG_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")

    figure_entry = {
        "title": title,
        "kind": kind,
        "path": out_path.name,
        "file_path": str(out_path),
        "data_uri": image_file_to_data_uri(out_path),
        "segment_context": segment_context
    }
    
    FIGURES.append(figure_entry)
    
    # Track segments for report organization
    if segment_context not in REPORT_SEGMENTS:
        REPORT_SEGMENTS[segment_context] = []
    REPORT_SEGMENTS[segment_context].append(figure_entry)

    if close:
        plt.close(fig)

    return out_path


def add_table(title: str, df: pd.DataFrame, max_rows: int = 20, segment_context: str = 'global'):
    table_entry = {
        'title': title, 
        'html': df.head(max_rows).to_html(index=False, classes='report-table'),
        'segment_context': segment_context
    }
    REPORT_TABLES.append(table_entry)


def monthify(series):
    return pd.to_datetime(series, errors='coerce').dt.to_period('M').astype(str)


def select_months(months, max_months=None):
    """Select months with optional trailing window limit"""
    months = [m for m in sorted(pd.Series(months).dropna().unique()) if m != 'NaT']
    if max_months is not None and len(months) > max_months:
        months = months[-max_months:]
    return months


def canonical_pair(x, y):
    return tuple(sorted((x, y)))


def top_pairs_from_corr(corr_df: pd.DataFrame, top_n: int = 6):
    tmp = corr_df.copy()
    tmp['abs_corr'] = tmp['correlation_value'].abs()
    tmp = tmp.sort_values('abs_corr', ascending=False)
    tmp['pair'] = tmp.apply(lambda r: canonical_pair(r['base_metric_x'], r['base_metric_y']), axis=1)
    tmp = tmp.drop_duplicates('pair')
    return [tuple(x) for x in tmp['pair'].head(top_n).tolist()]


def correlation_matrix_from_pairs(df: pd.DataFrame, metrics):
    mat = pd.DataFrame(np.eye(len(metrics)), index=metrics, columns=metrics)
    for _, row in df.iterrows():
        x = row['base_metric_x']
        y = row['base_metric_y']
        if x in mat.index and y in mat.columns:
            mat.loc[x, y] = row['correlation_value']
            mat.loc[y, x] = row['correlation_value']
    return mat


def build_density_matrix(pair_df: pd.DataFrame) -> np.ndarray:
    if pair_df.empty:
        return np.zeros((1, 1), dtype=float)
    x_max = int(pair_df['x_bin'].max())
    y_max = int(pair_df['y_bin'].max())
    mat = np.zeros((y_max + 1, x_max + 1), dtype=float)
    for _, row in pair_df.iterrows():
        mat[int(row['y_bin']), int(row['x_bin'])] += float(row['count'])
    return mat


def image_file_to_data_uri(path: Path) -> str:
    suffix = path.suffix.lower()
    mime_map = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".svg": "image/svg+xml",
    }
    mime_type = mime_map.get(suffix, "application/octet-stream")
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


# ========================================
# SEGMENTATION HELPERS
# ========================================

STRING_SEGMENT_DIMS = {"customer_segment", "product_segment", "region_rollup", "gen"}


def normalize_segment_value(value, segment_dim: str = None):
    """Normalize segment values for consistent comparisons across datasets."""
    if value is None or pd.isna(value):
        return None
    s = str(value).strip()
    if segment_dim in STRING_SEGMENT_DIMS:
        s = s.lower()
    return s


def sanitize_segment_value(value: str) -> str:
    """Sanitize segment value for safe filename usage"""
    if pd.isna(value):
        return 'null'
    return str(value).replace('/', '_').replace(' ', '_').replace('.', '_')


def discover_segment_values(df: pd.DataFrame, segment_dim: str, segment_col: str = None) -> List[str]:
    """Discover unique segment values for a dimension"""
    # For segmented datasets, use segment_dim/segment_value columns
    if 'segment_dim' in df.columns and 'segment_value' in df.columns:
        seg_df = df[df['segment_dim'] == segment_dim]
        if seg_df.empty:
            return []
        values = (
            seg_df['segment_value']
            .map(lambda x: normalize_segment_value(x, segment_dim))
            .dropna()
            .unique()
            .tolist()
        )
        return sorted(values)[:MAX_SEGMENT_VALUES_PER_DIM]

    # For raw_sample, use direct column
    if segment_col and segment_col in df.columns:
        values = (
            df[segment_col]
            .map(lambda x: normalize_segment_value(x, segment_dim))
            .dropna()
            .unique()
            .tolist()
        )
        return sorted(values)[:MAX_SEGMENT_VALUES_PER_DIM]

    return []


def filter_by_segment(df: pd.DataFrame, segment_dim: str, segment_value: str) -> pd.DataFrame:
    """Filter dataframe by segment dimension and value (normalized)."""
    target = normalize_segment_value(segment_value, segment_dim)
    if target is None:
        return df.iloc[0:0].copy()

    # For segmented datasets
    if 'segment_dim' in df.columns and 'segment_value' in df.columns:
        seg_mask = df['segment_dim'] == segment_dim
        val_mask = df['segment_value'].map(lambda x: normalize_segment_value(x, segment_dim)) == target
        return df[seg_mask & val_mask]

    # For raw_sample, use direct column
    if segment_dim in df.columns:
        val_mask = df[segment_dim].map(lambda x: normalize_segment_value(x, segment_dim)) == target
        return df[val_mask]

    return df.iloc[0:0].copy()


def get_segment_context_string(segment_dim: str = None, segment_value: str = None) -> str:
    """Get a readable segment context string"""
    if segment_dim is None or segment_value is None:
        return 'global'
    return f'{segment_dim}={sanitize_segment_value(segment_value)}'


def check_segment_data_availability(df: pd.DataFrame, min_rows: int = MIN_SEGMENT_ROWS) -> bool:
    """Check if segment has enough data for visualization"""
    return len(df) >= min_rows


def write_html_report(figures, tables, html_path: Path, summary_lines, segments):
    template = Template(HTML_TEMPLATE)
    html = template.render(
        figures=figures, 
        tables=tables, 
        summary_lines=summary_lines,
        segments=segments
    )
    html_path.write_text(html, encoding="utf-8")


def zip_png_pack(figures, zip_path: Path):
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fig in figures:
            zf.write(FIG_DIR / fig['path'], arcname=fig['path'])


# ========================================
# HTML TEMPLATE
# ========================================

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>DCGM EDA Visual Summary</title>
  <style>
    body {
      font-family: Arial, sans-serif;
      margin: 24px;
      color: #222;
      background: #fafafa;
    }
    h1, h2, h3, h4 {
      color: #111;
    }
    .summary {
      background: white;
      border: 1px solid #ddd;
      border-radius: 8px;
      padding: 16px;
      margin-bottom: 24px;
    }
    .segment-section {
      background: #f8f9fa;
      border-left: 4px solid #007bff;
      padding: 12px;
      margin: 20px 0;
    }
    .figure-card, .table-card {
      background: white;
      border: 1px solid #ddd;
      border-radius: 8px;
      padding: 16px;
      margin-bottom: 24px;
    }
    img {
      max-width: 100%;
      height: auto;
      display: block;
      margin-top: 12px;
      border: 1px solid #eee;
    }
    table.report-table {
      border-collapse: collapse;
      width: 100%;
      margin-top: 12px;
    }
    table.report-table th, table.report-table td {
      border: 1px solid #ccc;
      padding: 6px 10px;
      text-align: left;
    }
    table.report-table th {
      background: #f0f0f0;
    }
    .meta {
      color: #666;
      font-size: 13px;
    }
    .segment-label {
      background: #e9ecef;
      padding: 2px 6px;
      border-radius: 3px;
      font-size: 12px;
      color: #495057;
    }
  </style>
</head>
<body>
  <h1>DCGM EDA Visual Summary</h1>

  <div class="summary">
    <h2>Run Summary</h2>
    <ul>
      {% for line in summary_lines %}
      <li>{{ line }}</li>
      {% endfor %}
    </ul>
  </div>

  {% for segment_key, segment_figures in segments.items() %}
  <div class="segment-section">
    <h2>{% if segment_key == 'global' %}Global Views{% else %}Segment: {{ segment_key }}{% endif %}</h2>
    
    {% for fig in segment_figures %}
    <div class="figure-card">
      <h3>{{ fig.title }}</h3>
      <div class="meta">
        {{ fig.kind }}
        {% if fig.segment_context != 'global' %}
        <span class="segment-label">{{ fig.segment_context }}</span>
        {% endif %}
      </div>
      <img src="{{ fig.data_uri }}" alt="{{ fig.title }}">
    </div>
    {% endfor %}
  </div>
  {% endfor %}

  <h2>Tables</h2>
  {% for table in tables %}
  <div class="table-card">
    <h3>{{ table.title }}</h3>
    {% if table.segment_context != 'global' %}
    <span class="segment-label">{{ table.segment_context }}</span>
    {% endif %}
    {{ table.html | safe }}
  </div>
  {% endfor %}
</body>
</html>
"""

# ========================================
# VISUALIZATION FUNCTIONS FOR SEGMENTATION
# ========================================

def render_core_density_heatmaps(density_df, segment_dim=None, segment_value=None):
    """Render core metric density heatmaps"""
    segment_context = get_segment_context_string(segment_dim, segment_value)
    
    # Filter data if segmented
    if segment_dim and segment_value:
        density_df = filter_by_segment(density_df, segment_dim, segment_value)
        if not check_segment_data_availability(density_df):
            print(f'  Skipping density heatmaps for {segment_context}: insufficient data')
            return
    
    core_pairs = list(combinations(CORE_AVAILABLE, 2))
    if not core_pairs:
        return
        
    ncols = 3
    nrows = math.ceil(len(core_pairs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.6 * nrows))
    axes = np.array(axes).reshape(-1)
    
    for ax, (mx, my) in zip(axes, core_pairs):
        pair_df = density_df[(density_df['metric_x'] == mx) & (density_df['metric_y'] == my)].copy()
        if pair_df.empty:
            pair_df = density_df[(density_df['metric_x'] == my) & (density_df['metric_y'] == mx)].copy()
        mat = build_density_matrix(pair_df)
        im = ax.imshow(np.log1p(mat), origin='lower', aspect='auto', cmap='magma')
        ax.set_title(f'{mx} vs {my}')
        ax.set_xlabel(f'{mx} bin')
        ax.set_ylabel(f'{my} bin')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    for ax in axes[len(core_pairs):]:
        ax.axis('off')
    
    title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
    fig.suptitle(f'Core Metric Density Heat Maps{title_suffix}', y=1.02)
    fig.tight_layout()
    
    filename = f'01_core_density_{sanitize_segment_value(segment_context)}.png'
    save_figure(fig, filename, f'Core metric heat maps{title_suffix}', 'Static density heat map grid', segment_context)


def render_distribution_charts(raw_df, metrics, segment_dim=None, segment_value=None, chart_type='histogram'):
    """Render distribution charts (histograms or boxplots)"""
    segment_context = get_segment_context_string(segment_dim, segment_value)
    
    # Filter data if segmented
    if segment_dim and segment_value:
        raw_df = filter_by_segment(raw_df, segment_dim, segment_value)
        if not check_segment_data_availability(raw_df):
            print(f'  Skipping {chart_type} for {segment_context}: insufficient data')
            return
    
    if chart_type == 'histogram':
        ncols = 3
        nrows = math.ceil(len(metrics) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.8 * nrows))
        axes = np.array(axes).reshape(-1)
        
        for ax, metric in zip(axes, metrics):
            sns.histplot(raw_df[metric].dropna(), bins=40, kde=False, ax=ax, color='#4C78A8')
            ax.set_title(metric)
            ax.set_xlabel(metric)
            ax.set_ylabel('count')
        
        for ax in axes[len(metrics):]:
            ax.axis('off')
        
        title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
        fig.suptitle(f'Metric Distributions{title_suffix}', y=1.02)
        fig.tight_layout()
        
        filename = f'03_histograms_{sanitize_segment_value(segment_context)}.png'
        save_figure(fig, filename, f'Metric histograms{title_suffix}', 'Histogram grid', segment_context)
    
    elif chart_type == 'boxplot':
        # Use BOXPLOT_METRICS to exclude specified metrics
        plot_metrics = [m for m in metrics if m in BOXPLOT_METRICS]
        if not plot_metrics:
            return
            
        melted = raw_df[plot_metrics].melt(var_name='metric', value_name='value').dropna()
        fig, ax = plt.subplots(figsize=(max(12, len(plot_metrics) * 0.8), 6.5))
        sns.boxplot(data=melted, x='metric', y='value', ax=ax, showfliers=False)
        
        title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
        ax.set_title(f'Box-and-whisker summary{title_suffix}')
        ax.set_xlabel('metric')
        ax.set_ylabel('value')
        ax.tick_params(axis='x', rotation=45)
        fig.tight_layout()
        
        filename = f'04_boxplots_{sanitize_segment_value(segment_context)}.png'
        save_figure(fig, filename, f'Metric box-and-whisker{title_suffix}', 'Static boxplot summary', segment_context)


def render_correlation_heatmap(corr_df, corr_group, segment_dim=None, segment_value=None):
    """Render correlation heatmap for a specific group (p50, p95)"""
    segment_context = get_segment_context_string(segment_dim, segment_value)
    
    # Filter data if segmented
    if segment_dim and segment_value:
        corr_df = filter_by_segment(corr_df, segment_dim, segment_value)
    
    static_corr = corr_df[
        (corr_df['method'] == DEFAULT_CORR_METHOD) &
        (corr_df['summary_feature_group'] == corr_group) &
        (corr_df['base_metric_x'].isin(CORR_METRICS)) &
        (corr_df['base_metric_y'].isin(CORR_METRICS))
    ].copy()
    
    if static_corr.empty:
        print(f'  No correlation data for {corr_group} in {segment_context}')
        return
    
    corr_mat = correlation_matrix_from_pairs(static_corr, CORR_METRICS)
    if corr_mat.empty or corr_mat.isna().all().all():
        print(f'  No usable correlation matrix for {corr_group} in {segment_context}')
        return

    fig, ax = plt.subplots(figsize=(9.5, 8.2))
    sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
    
    title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
    ax.set_title(f'{DEFAULT_CORR_METHOD.title()} correlation heatmap ({corr_group}){title_suffix}')
    fig.tight_layout()
    
    filename = f'06_corr_heatmap_{corr_group}_{sanitize_segment_value(segment_context)}.png'
    save_figure(fig, filename, f'Correlation heatmap ({corr_group}){title_suffix}', f'Static correlation heatmap ({corr_group})', segment_context)


# ========================================
# ADDITIONAL RENDER HELPERS (GLOBAL + SEGMENTED)
# ========================================

def render_monthly_density_drift(density_over_time_df, pairs, months, segment_dim=None, segment_value=None):
    segment_context = get_segment_context_string(segment_dim, segment_value)

    if segment_dim and segment_value:
        density_over_time_df = filter_by_segment(density_over_time_df, segment_dim, segment_value)
        if density_over_time_df.empty:
            print(f'  Skipping monthly density drift for {segment_context}: no segmented data')
            return

    if not pairs:
        return

    for mx, my in pairs:
        pair_df = density_over_time_df[(density_over_time_df['metric_x'] == mx) & (density_over_time_df['metric_y'] == my)].copy()
        if pair_df.empty:
            pair_df = density_over_time_df[(density_over_time_df['metric_x'] == my) & (density_over_time_df['metric_y'] == mx)].copy()

        pair_df = pair_df[pair_df['month_period'].isin(months)]
        pair_months = [m for m in months if m in set(pair_df['month_period'])]
        if not pair_months:
            continue

        display_months = pair_months[-9:] if len(pair_months) > 9 else pair_months
        fig, axes = plt.subplots(1, len(display_months), figsize=(5 * len(display_months), 4.2), squeeze=False)
        axes = axes[0]
        for ax, month in zip(axes, display_months):
            month_df = pair_df[pair_df['month_period'] == month]
            mat = build_density_matrix(month_df)
            im = ax.imshow(np.log1p(mat), origin='lower', aspect='auto', cmap='viridis')
            ax.set_title(month)
            ax.set_xlabel(f'{mx} bin')
            ax.set_ylabel(f'{my} bin')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
        fig.suptitle(f'Monthly density drift: {mx} vs {my} (trailing {len(months)} months){title_suffix}', y=1.05)
        fig.tight_layout()
        filename = f'02_monthly_density_{mx}_vs_{my}_{sanitize_segment_value(segment_context)}.png'
        save_figure(fig, filename, f'Monthly density drift: {mx} vs {my}{title_suffix}', 'Monthly small multiples', segment_context)


def render_monthly_distribution_drift(raw_df, metrics, months, segment_dim=None, segment_value=None):
    segment_context = get_segment_context_string(segment_dim, segment_value)

    if segment_dim and segment_value:
        raw_df = filter_by_segment(raw_df, segment_dim, segment_value)
        if not check_segment_data_availability(raw_df):
            print(f'  Skipping monthly distribution drift for {segment_context}: insufficient data')
            return

    if not metrics or not months:
        return

    melted = raw_df[raw_df['month_period'].isin(months)][['month_period'] + metrics].melt(
        id_vars='month_period', var_name='metric', value_name='value'
    ).dropna()
    if melted.empty:
        return

    g = sns.catplot(
        data=melted,
        x='month_period',
        y='value',
        col='metric',
        col_wrap=3,
        kind='box',
        sharey=False,
        showfliers=False,
        height=3.4,
        aspect=1.4,
    )
    g.set_titles('{col_name}')
    for ax in g.axes.flatten():
        ax.tick_params(axis='x', rotation=45)
    title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
    g.fig.suptitle(f'Monthly distribution drift (trailing {len(months)} months){title_suffix}', y=1.02)
    filename = f'05_monthly_metric_boxplots_{sanitize_segment_value(segment_context)}.png'
    save_figure(g.fig, filename, f'Monthly distribution drift{title_suffix}', 'Monthly small multiples', segment_context)


def render_monthly_correlation_drift(corr_over_time_df, corr_group, months, segment_dim=None, segment_value=None):
    segment_context = get_segment_context_string(segment_dim, segment_value)

    if segment_dim and segment_value:
        corr_over_time_df = filter_by_segment(corr_over_time_df, segment_dim, segment_value)
        if corr_over_time_df.empty:
            print(f'  Skipping monthly correlation drift for {segment_context}: no segmented data')
            return

    monthly_corr = corr_over_time_df[
        (corr_over_time_df['method'] == DEFAULT_CORR_METHOD)
        & (corr_over_time_df['summary_percentile'] == corr_group)
        & (corr_over_time_df['base_metric_x'].isin(CORR_METRICS))
        & (corr_over_time_df['base_metric_y'].isin(CORR_METRICS))
        & (corr_over_time_df['month_period'].isin(months))
    ].copy()

    if monthly_corr.empty:
        print(f'  No monthly correlation data for {corr_group} in {segment_context}')
        return

    monthly_corr = monthly_corr.groupby(['month_period', 'base_metric_x', 'base_metric_y'], as_index=False)['correlation_value'].mean()
    corr_months = [m for m in months if m in set(monthly_corr['month_period'])]
    if not corr_months:
        return

    display_months = corr_months[-9:] if len(corr_months) > 9 else corr_months
    month_mats = {}
    for month in display_months:
        month_df = monthly_corr[monthly_corr['month_period'] == month]
        mat = correlation_matrix_from_pairs(month_df, CORR_METRICS)
        if mat.empty or mat.isna().all().all():
            continue
        month_mats[month] = mat

    if not month_mats:
        print(f'  No usable correlation matrix for {corr_group} in {segment_context}')
        return

    fig, axes = plt.subplots(1, len(month_mats), figsize=(5.5 * len(month_mats), 4.8), squeeze=False)
    axes = axes[0]
    for ax, month in zip(axes, month_mats):
        mat = month_mats[month]
        sns.heatmap(mat, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, cbar=ax is axes[-1])
        ax.set_title(month)
    title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
    fig.suptitle(f'Monthly correlation drift ({DEFAULT_CORR_METHOD}, {corr_group}) - trailing {len(months)} months{title_suffix}', y=1.04)
    fig.tight_layout()
    filename = f'07_monthly_correlation_heatmaps_{corr_group}_{sanitize_segment_value(segment_context)}.png'
    save_figure(fig, filename, f'Monthly correlation drift heatmaps ({corr_group}){title_suffix}', f'Monthly small multiples ({corr_group})', segment_context)


def get_scatter_pairs(corr_df, corr_group='p50', segment_dim=None, segment_value=None, top_n=7):
    if corr_df.empty:
        return list(combinations(CORE_EXTENDED[:4], 2))[:top_n]

    if segment_dim and segment_value:
        corr_df = filter_by_segment(corr_df, segment_dim, segment_value)

    static_corr = corr_df[
        (corr_df['method'] == DEFAULT_CORR_METHOD)
        & (corr_df['summary_feature_group'] == corr_group)
        & (corr_df['base_metric_x'].isin(CORR_METRICS))
        & (corr_df['base_metric_y'].isin(CORR_METRICS))
    ].copy()

    if static_corr.empty:
        return list(combinations(CORE_EXTENDED[:4], 2))[:top_n]

    return top_pairs_from_corr(static_corr, top_n=top_n)


def render_pairwise_scatter(raw_df, pairs, segment_dim=None, segment_value=None):
    segment_context = get_segment_context_string(segment_dim, segment_value)

    if segment_dim and segment_value:
        raw_df = filter_by_segment(raw_df, segment_dim, segment_value)
        if not check_segment_data_availability(raw_df):
            print(f'  Skipping pairwise scatter for {segment_context}: insufficient data')
            return

    if not pairs:
        return

    # Build unique metric set from selected pairs
    metrics = []
    for mx, my in pairs:
        if mx in raw_df.columns and my in raw_df.columns:
            metrics.extend([mx, my])
    metrics = list(dict.fromkeys(metrics))  # preserve order, de-dupe

    if len(metrics) < 2:
        print(f'  Skipping pairwise scatter for {segment_context}: not enough metrics')
        return

    pair_df = raw_df[metrics].dropna()
    if pair_df.empty:
        print(f'  Skipping pairwise scatter for {segment_context}: no complete rows')
        return

    if len(pair_df) > MAX_PAIRPLOT_ROWS:
        pair_df = pair_df.sample(MAX_PAIRPLOT_ROWS, random_state=SEED)

    title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''

    g = sns.pairplot(
        pair_df,
        corner=True,          # lower triangle only; remove if you want full square matrix
        diag_kind='hist',     # or 'kde'
        plot_kws={'s': 10, 'alpha': 0.25},
        diag_kws={'bins': 30}
    )

    g.fig.suptitle(f'Pairwise scatter matrix{title_suffix}', y=1.02)
    g.fig.tight_layout()

    filename = f'08_pairwise_scatter_matrix_{sanitize_segment_value(segment_context)}.png'
    save_figure(
        g.fig,
        filename,
        f'Pairwise scatter matrix{title_suffix}',
        'Scatterplot matrix',
        segment_context
    )



def render_monthly_scatter_drift(raw_df, pairs, months, segment_dim=None, segment_value=None):
    segment_context = get_segment_context_string(segment_dim, segment_value)

    if segment_dim and segment_value:
        raw_df = filter_by_segment(raw_df, segment_dim, segment_value)
        if not check_segment_data_availability(raw_df):
            print(f'  Skipping monthly scatter drift for {segment_context}: insufficient data')
            return

    for mx, my in pairs:
        if mx not in raw_df.columns or my not in raw_df.columns:
            continue

        pair_months = [m for m in months if m in set(raw_df['month_period'])]
        if not pair_months:
            continue

        display_months = pair_months[-9:] if len(pair_months) > 9 else pair_months
        fig, axes = plt.subplots(1, len(display_months), figsize=(5 * len(display_months), 4.2), squeeze=False)
        axes = axes[0]
        for ax, month in zip(axes, display_months):
            month_pdf = raw_df[raw_df['month_period'] == month][[mx, my]].dropna()
            if len(month_pdf) > MAX_SCATTER_ROWS_PER_MONTH:
                month_pdf = month_pdf.sample(MAX_SCATTER_ROWS_PER_MONTH, random_state=SEED)
            sns.scatterplot(data=month_pdf, x=mx, y=my, s=10, alpha=0.25, ax=ax)
            ax.set_title(month)
        title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
        fig.suptitle(f'Monthly scatter drift: {mx} vs {my} (trailing {len(months)} months){title_suffix}', y=1.05)
        fig.tight_layout()
        filename = f'09_monthly_scatter_{mx}_vs_{my}_{sanitize_segment_value(segment_context)}.png'
        save_figure(fig, filename, f'Monthly scatter drift: {mx} vs {my}{title_suffix}', 'Monthly small multiples', segment_context)


# Override to add drift-score sample output

def render_iqr_scaled_drift_heatmap(raw_df, metrics, months, segment_dim=None, segment_value=None):
    segment_context = get_segment_context_string(segment_dim, segment_value)

    if segment_dim and segment_value:
        raw_df = filter_by_segment(raw_df, segment_dim, segment_value)
        if not check_segment_data_availability(raw_df):
            print(f'  Skipping drift heatmap for {segment_context}: insufficient data')
            return

    raw_df = raw_df[raw_df['month_period'].isin(months)]

    metric_month = raw_df[['month_period'] + metrics].melt(
        id_vars='month_period', var_name='metric', value_name='value'
    ).dropna()

    if metric_month.empty:
        print(f'  No data for drift heatmap in {segment_context}')
        return

    baseline_month = months[0]
    baseline_stats = metric_month[metric_month['month_period'] == baseline_month].groupby('metric')['value'].agg([
        ('baseline_median', 'median'),
        ('baseline_q1', lambda x: x.quantile(0.25)),
        ('baseline_q3', lambda x: x.quantile(0.75))
    ])
    baseline_stats['baseline_iqr'] = baseline_stats['baseline_q3'] - baseline_stats['baseline_q1']

    epsilon = 1e-6
    baseline_stats['baseline_iqr'] = baseline_stats['baseline_iqr'].clip(lower=epsilon)

    current = metric_month.groupby(['month_period', 'metric'])['value'].median().reset_index()
    current = current.merge(baseline_stats[['baseline_median', 'baseline_iqr']], left_on='metric', right_index=True, how='left')

    current['drift_score'] = (current['value'] - current['baseline_median']) / current['baseline_iqr']

    pivot = current.pivot(index='metric', columns='month_period', values='drift_score')
    available_metrics = [m for m in metrics if m in pivot.index]
    if available_metrics:
        pivot = pivot.reindex(available_metrics)

    fig, ax = plt.subplots(figsize=(max(10, len(months) * 1.2), max(8, len(available_metrics) * 0.45)))
    sns.heatmap(pivot, cmap='coolwarm', center=0, ax=ax)

    title_suffix = f' [{segment_context}]' if segment_context != 'global' else ''
    ax.set_title(f'IQR-scaled metric drift vs baseline ({baseline_month}){title_suffix}')
    fig.tight_layout()

    filename = f'11_drift_heatmap_{sanitize_segment_value(segment_context)}.png'
    save_figure(fig, filename, f'IQR-scaled drift heatmap{title_suffix}', 'IQR-scaled drift summary', segment_context)

    if segment_context == 'global':
        print('\nSample IQR-scaled drift calculations:')
        sample_metrics = available_metrics[:3] if len(available_metrics) > 3 else available_metrics
        sample_month = months[-1] if months else None
        for metric in sample_metrics:
            if metric in baseline_stats.index:
                median = baseline_stats.loc[metric, 'baseline_median']
                iqr = baseline_stats.loc[metric, 'baseline_iqr']
                drift_val = None
                if sample_month is not None:
                    match = current[(current['month_period'] == sample_month) & (current['metric'] == metric)]
                    if not match.empty:
                        drift_val = float(match['drift_score'].iloc[0])
                drift_msg = f', drift_score@{sample_month}={drift_val:.3f}' if drift_val is not None else ''
                print(f'  {metric}: baseline_median={median:.3f}, baseline_iqr={iqr:.3f}{drift_msg}')


def write_split_html_reports(figures, tables, html_dir: Path, summary_lines, segments):
    """Write separate HTML reports for each segment to reduce file size."""
    
    # Create index page with navigation
    index_template = Template('''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DCGM EDA Report Index</title>
    <style>
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
            line-height: 1.6;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
        }
        h1 { color: #2c3e50; }
        .summary {
            background: #f9f9f9;
            padding: 15px;
            border-radius: 5px;
            margin: 20px 0;
        }
        .segment-links {
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(300px, 1fr));
            gap: 15px;
            margin: 20px 0;
        }
        .segment-card {
            background: white;
            border: 1px solid #ddd;
            border-radius: 5px;
            padding: 15px;
            text-decoration: none;
            color: inherit;
            transition: box-shadow 0.3s;
        }
        .segment-card:hover {
            box-shadow: 0 4px 8px rgba(0,0,0,0.1);
        }
        .segment-card h3 {
            margin: 0 0 10px 0;
            color: #3498db;
        }
        .segment-stats {
            font-size: 14px;
            color: #666;
        }
        .global-report {
            background: #e8f4f8;
            border-color: #3498db;
        }
    </style>
</head>
<body>
    <h1>DCGM EDA Visual Summary - Report Index</h1>
    
    <div class="summary">
        <h2>Summary</h2>
        <ul>
        {% for line in summary_lines %}
            <li>{{ line }}</li>
        {% endfor %}
        </ul>
    </div>
    
    <h2>Available Reports</h2>
    <div class="segment-links">
        {% for segment_name, segment_figures in segments.items() %}
        <a href="{{ segment_name | replace('/', '_') | replace(' ', '_') }}.html" class="segment-card {% if segment_name == 'global' %}global-report{% endif %}">
            <h3>{{ segment_name | title if segment_name == 'global' else segment_name }}</h3>
            <div class="segment-stats">
                <strong>{{ segment_figures | length }}</strong> visualizations<br>
                {% if segment_name == 'global' %}
                Complete dataset analysis
                {% else %}
                Segmented analysis
                {% endif %}
            </div>
        </a>
        {% endfor %}
    </div>
    
    <div style="margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; color: #666; font-size: 14px;">
        <p>Generated: {{ timestamp }}</p>
        <p>Reports split to reduce file size. Each segment has its own HTML file.</p>
    </div>
</body>
</html>
    ''')
    
    # Write index page
    index_path = html_dir / 'index.html'
    index_html = index_template.render(
        segments=segments,
        summary_lines=summary_lines,
        timestamp=datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    )
    index_path.write_text(index_html, encoding='utf-8')
    print(f'Created index: {index_path}')
    
    # Write separate HTML for each segment
    for segment_name, segment_figures in segments.items():
        # Get tables for this segment
        segment_tables = [t for t in tables if t.get('segment_context', 'global') == segment_name]
        
        # Clean filename
        safe_filename = segment_name.replace('/', '_').replace(' ', '_') + '.html'
        segment_path = html_dir / safe_filename
        
        # Add navigation back to index
        nav_html = '''
        <div style="margin-bottom: 20px; padding: 10px; background: #f0f0f0; border-radius: 5px;">
            <a href="index.html" style="text-decoration: none; color: #3498db;">← Back to Index</a>
            | <strong>{{ segment_name }}</strong>
        </div>
        '''
        
        # Modify template to include navigation
        segment_template_str = HTML_TEMPLATE.replace(
            '<h1>DCGM EDA Visual Summary</h1>',
            '<h1>DCGM EDA Visual Summary - {{ segment_name }}</h1>\n' + nav_html
        )
        
        segment_template = Template(segment_template_str)
        segment_html = segment_template.render(
            figures=segment_figures,
            tables=segment_tables,
            summary_lines=summary_lines,
            segments={segment_name: segment_figures},  # Only this segment
            segment_name=segment_name
        )
        segment_path.write_text(segment_html, encoding='utf-8')
        print(f'Created segment report: {segment_path} ({len(segment_figures)} figures)')
    
    print(f'\nAll reports written to: {html_dir}')
    print(f'Main index: {index_path}')



In [4]:
# ========================================
# LOAD PARQUET EXPORTS
# ========================================
print('=' * 60)
print('LOADING PARQUET EXPORTS')
print('=' * 60)

resolved_root = discover_export_root(PARQUET_ROOT, EXPORT_TAG)
manifest_file = resolved_root / 'manifest.json'
manifest = json.loads(manifest_file.read_text())

ALL_MONTHS = discover_available_months(resolved_root, 'raw_sample', ['month_period', 'datestamp'])
MONTHS = select_months(ALL_MONTHS, max_months=MAX_MONTHS)

RAW_SEGMENT_COLS = ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
RAW_BASE_COLS = ['datestamp', 'month_period'] + RAW_SEGMENT_COLS
RAW_METRIC_COLS = list(dict.fromkeys(CORE_METRICS + EXTENDED_METRICS + INDEX_METRICS))
RAW_SAMPLE_COLS = RAW_BASE_COLS + RAW_METRIC_COLS

CORR_SUMMARY_COLS = [
    'summary_feature_group', 'summary_percentile', 'method',
    'base_metric_x', 'base_metric_y', 'correlation_value',
    'segment_dim', 'segment_value'
]

CORR_OVER_TIME_COLS = [
    'time_bucket', 'month_period', 'summary_percentile', 'method',
    'base_metric_x', 'base_metric_y', 'correlation_value',
    'segment_dim', 'segment_value'
]

DENSITY_BINS_COLS = [
    'metric_x', 'metric_y', 'x_bin', 'y_bin', 'count',
    'segment_dim', 'segment_value'
]

DENSITY_OVER_TIME_COLS = [
    'time_bucket', 'month_period',
    'metric_x', 'metric_y', 'x_bin', 'y_bin', 'count',
    'segment_dim', 'segment_value'
]

BIN_METADATA_COLS = [
    'metric_name', 'bin_number', 'bin_min', 'bin_max',
    'num_bins', 'percentile_low', 'percentile_high',
    'segment_dim', 'segment_value'
]

TREND_SUMMARY_COLS = [
    'time_bucket', 'month_period', 'metric_name', 'value',
    'segment_dim', 'segment_value'
]

raw_sample_pdf = load_parquet_dataset(
    resolved_root,
    'raw_sample',
    columns=RAW_SAMPLE_COLS,
    month_filter=MONTHS,
    month_col='datestamp',
)

corr_summary_pdf = load_parquet_dataset(
    resolved_root,
    'corr_summary',
    columns=CORR_SUMMARY_COLS,
)

corr_over_time_pdf = load_parquet_dataset(
    resolved_root,
    'corr_over_time',
    columns=CORR_OVER_TIME_COLS,
    month_filter=MONTHS,
    month_col='month_period',
)

density_bins_pdf = load_parquet_dataset(
    resolved_root,
    'density_bins',
    columns=DENSITY_BINS_COLS,
)

density_over_time_pdf = load_parquet_dataset(
    resolved_root,
    'density_over_time',
    columns=DENSITY_OVER_TIME_COLS,
    month_filter=MONTHS,
    month_col='month_period',
)

bin_metadata_pdf = load_parquet_dataset(
    resolved_root,
    'bin_metadata',
    columns=BIN_METADATA_COLS,
)

trend_summary_pdf = load_parquet_dataset(
    resolved_root,
    'trend_summary',
    columns=TREND_SUMMARY_COLS,
    month_filter=MONTHS,
    month_col='month_period',
)

MAX_RAW_SAMPLE_ROWS = 2_000_000
if len(raw_sample_pdf) > MAX_RAW_SAMPLE_ROWS:
    raw_sample_pdf = raw_sample_pdf.sample(MAX_RAW_SAMPLE_ROWS, random_state=SEED).copy()
    print(f'Downsampled raw_sample to {len(raw_sample_pdf):,} rows for local visualization stability')

raw_sample_pdf['month_period'] = monthify(raw_sample_pdf.get('month_period', raw_sample_pdf['datestamp']))
corr_over_time_pdf['month_period'] = monthify(corr_over_time_pdf.get('month_period', corr_over_time_pdf['time_bucket']))
density_over_time_pdf['month_period'] = monthify(density_over_time_pdf.get('month_period', density_over_time_pdf['time_bucket']))
trend_summary_pdf['month_period'] = monthify(trend_summary_pdf.get('month_period', trend_summary_pdf['time_bucket']))

print(f'Resolved export root: {resolved_root}')
print(f'Trailing month window: {len(MONTHS)} months ({MONTHS[0] if MONTHS else "N/A"} to {MONTHS[-1] if MONTHS else "N/A"})')
for name, pdf in [
    ('raw_sample', raw_sample_pdf),
    ('corr_summary', corr_summary_pdf),
    ('corr_over_time', corr_over_time_pdf),
    ('density_bins', density_bins_pdf),
    ('density_over_time', density_over_time_pdf),
    ('bin_metadata', bin_metadata_pdf),
    ('trend_summary', trend_summary_pdf),
]:
    print(f'  {name:18} -> {len(pdf):,} rows')
print('=' * 60)


LOADING PARQUET EXPORTS
Resolved export root: /home/coreweave/dcgm_eda_parquet/20260625_164113
Trailing month window: 15 months (2025-03 to 2026-05)
  raw_sample         -> 405,459 rows
  corr_summary       -> 8,658 rows
  corr_over_time     -> 3,962,070 rows
  density_bins       -> 1,100,377 rows
  density_over_time  -> 319,280,368 rows
  bin_metadata       -> 6,272 rows
  trend_summary      -> 283,275 rows


In [5]:
# ========================================
# DATA CONTRACT VALIDATION
# ========================================
print('=' * 60)
print('DATA CONTRACT VALIDATION')
print('=' * 60)

# Check for segmentation fields in segmented datasets
segmented_datasets = ['corr_summary', 'corr_over_time', 'density_bins', 'density_over_time', 'trend_summary']
for name, df in [
    ('corr_summary', corr_summary_pdf),
    ('corr_over_time', corr_over_time_pdf),
    ('density_bins', density_bins_pdf),
    ('density_over_time', density_over_time_pdf),
    ('trend_summary', trend_summary_pdf)
]:
    has_segment_dim = 'segment_dim' in df.columns
    has_segment_value = 'segment_value' in df.columns
    print(f'{name:18} - segment_dim: {"✓" if has_segment_dim else "✗"}, segment_value: {"✓" if has_segment_value else "✗"}')

print()
print('Checking raw_sample for segment source fields:')
for field in SEGMENT_DIMS_TO_RENDER:
    has_field = field in raw_sample_pdf.columns
    if has_field:
        unique_count = raw_sample_pdf[field].nunique()
        print(f'  {field:20} - {"✓" if has_field else "✗"} ({unique_count} unique values)')
    else:
        print(f'  {field:20} - ✗ (not found)')

print('=' * 60)


DATA CONTRACT VALIDATION
corr_summary       - segment_dim: ✓, segment_value: ✓
corr_over_time     - segment_dim: ✓, segment_value: ✓
density_bins       - segment_dim: ✓, segment_value: ✓
density_over_time  - segment_dim: ✓, segment_value: ✓
trend_summary      - segment_dim: ✓, segment_value: ✓

Checking raw_sample for segment source fields:
  gen                  - ✓ (2 unique values)
  region_rollup        - ✓ (2 unique values)
  customer_segment     - ✓ (4 unique values)
  product_segment      - ✓ (3 unique values)
  is_training          - ✓ (2 unique values)


In [6]:
# ========================================
# CHECK FOR POWER_PCT_MAX_INDEX METRIC
# ========================================
print('=' * 60)
print('CHECKING FOR POWER_PCT_MAX_INDEX METRIC')
print('=' * 60)

# Check if the metric exists in raw_sample_pdf
if 'power_pct_max_index' in raw_sample_pdf.columns:
    print('✓ power_pct_max_index found in raw_sample_pdf')
    print(f'  - Non-null values: {raw_sample_pdf["power_pct_max_index"].notna().sum():,}')
    print(f'  - Null values: {raw_sample_pdf["power_pct_max_index"].isna().sum():,}')
    print(f'  - Min value: {raw_sample_pdf["power_pct_max_index"].min():.3f}')
    print(f'  - Max value: {raw_sample_pdf["power_pct_max_index"].max():.3f}')
    print(f'  - Mean value: {raw_sample_pdf["power_pct_max_index"].mean():.3f}')
    print(f'  - Median value: {raw_sample_pdf["power_pct_max_index"].median():.3f}')
else:
    print('✗ power_pct_max_index NOT found in raw_sample_pdf')
    print('Available columns:', list(raw_sample_pdf.columns))

print('=' * 60)


CHECKING FOR POWER_PCT_MAX_INDEX METRIC
✓ power_pct_max_index found in raw_sample_pdf
  - Non-null values: 405,459
  - Null values: 0
  - Min value: 0.004
  - Max value: 3.092
  - Mean value: 0.321
  - Median value: 0.257


In [7]:
# ========================================
# METRIC RESOLUTION
# ========================================
ALL_RAW_METRICS = [m for m in CORE_METRICS + EXTENDED_METRICS + INDEX_METRICS if m in raw_sample_pdf.columns]
CORE_AVAILABLE = [m for m in CORE_METRICS if m in raw_sample_pdf.columns]
EXTENDED_AVAILABLE = [m for m in EXTENDED_METRICS if m in raw_sample_pdf.columns]
INDEX_AVAILABLE = [m for m in INDEX_METRICS if m in raw_sample_pdf.columns]
CORE_EXTENDED = [m for m in CORE_METRICS + EXTENDED_METRICS if m in raw_sample_pdf.columns]

# Metrics for box plots (excluding specified metrics)
BOXPLOT_METRICS = [m for m in ALL_RAW_METRICS if m not in BOXPLOT_EXCLUDE_METRICS]

corr_base_metrics = sorted(set(corr_summary_pdf['base_metric_x']).union(set(corr_summary_pdf['base_metric_y'])))
CORR_METRICS = [m for m in CORE_EXTENDED if m in corr_base_metrics]

# Select trailing months
ALL_MONTHS = select_months(raw_sample_pdf['month_period'], max_months=None)
MONTHS = select_months(raw_sample_pdf['month_period'], max_months=MAX_MONTHS)

print('=' * 60)
print('RESOLVED METRICS AND TIME WINDOWS')
print('=' * 60)
print(f'Core metrics         : {CORE_AVAILABLE}')
print(f'Extended metrics     : {EXTENDED_AVAILABLE}')
print(f'Index metrics        : {INDEX_AVAILABLE}')
print(f'Boxplot metrics      : {BOXPLOT_METRICS}')
print(f'Corr heatmap set     : {CORR_METRICS}')
print(f'All month periods    : {len(ALL_MONTHS)} months ({ALL_MONTHS[0] if ALL_MONTHS else "N/A"} to {ALL_MONTHS[-1] if ALL_MONTHS else "N/A"})')
print(f'Trailing month window: {len(MONTHS)} months ({MONTHS[0] if MONTHS else "N/A"} to {MONTHS[-1] if MONTHS else "N/A"})')
print('=' * 60)


RESOLVED METRICS AND TIME WINDOWS
Core metrics         : ['tensor_util', 'gpu_util', 'tensor_tflops', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active']
Extended metrics     : ['dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy']
Index metrics        : ['compute_occupancy_index', 'memory_boundness_index', 'memory_intensity_index', 'power_pct_max_index', 'compute_pct_max_index', 'TFLOPS_per_watt_efficiency']
Boxplot metrics      : ['tensor_util', 'gpu_util', 'sm_active', 'dram_active', 'mem_copy_util', 'sm_occupancy', 'compute_occupancy_index', 'memory_boundness_index', 'memory_intensity_index', 'power_pct_max_index', 'compute_pct_max_index', 'TFLOPS_per_watt_efficiency']
Corr heatmap set     : ['tensor_util', 'gpu_util', 'tensor_tflops_sm', 'chip_power', 'redfish_power', 'sm_active', 'dram_active', 'mem_copy_util', 'vram_usage', 'sm_clock', 'sm_occupancy']
All month periods    : 15 months (2025-03 to 2026-05)
Trailing month window: 15 months (2025-03 t

In [8]:
# ========================================
# SEGMENT DISCOVERY (CANONICAL)
# ========================================
print('=' * 60)
print('SEGMENT DISCOVERY')
print('=' * 60)

segmented_frames = []
for df in [corr_summary_pdf, corr_over_time_pdf, density_bins_pdf, density_over_time_pdf, trend_summary_pdf]:
    if 'segment_dim' in df.columns and 'segment_value' in df.columns:
        segmented_frames.append(df[['segment_dim', 'segment_value']].copy())

if segmented_frames:
    segmented_pairs = pd.concat(segmented_frames, ignore_index=True)
    segmented_pairs = segmented_pairs.dropna()
else:
    segmented_pairs = pd.DataFrame(columns=['segment_dim', 'segment_value'])

available_segments = []
available_segments_by_dim = {}

print('Segment discovery rules:')
print(f'  - dimensions considered: {SEGMENT_DIMS_TO_RENDER}')
print(f'  - max values per dimension: {MAX_SEGMENT_VALUES_PER_DIM}')
print(f'  - min rows per segment (raw_sample only): {MIN_SEGMENT_ROWS}')
print('  - null/blank values are skipped')

for dim in SEGMENT_DIMS_TO_RENDER:
    values = []
    if not segmented_pairs.empty:
        values += (
            segmented_pairs.loc[segmented_pairs['segment_dim'] == dim, 'segment_value']
            .dropna()
            .map(lambda x: normalize_segment_value(x, dim))
            .dropna()
            .tolist()
        )
    if dim in raw_sample_pdf.columns:
        values += (
            raw_sample_pdf[dim]
            .dropna()
            .map(lambda x: normalize_segment_value(x, dim))
            .dropna()
            .tolist()
        )

    values = [v for v in values if str(v).strip() != '' and str(v).lower() != 'nan']
    if not values:
        continue

    unique_values = pd.Series(values).dropna().unique().tolist()

    if dim in raw_sample_pdf.columns:
        counts = (
            raw_sample_pdf[dim]
            .dropna()
            .map(lambda x: normalize_segment_value(x, dim))
            .value_counts()
        )
        filtered = [v for v in unique_values if counts.get(v, 0) >= MIN_SEGMENT_ROWS]
        ordered = sorted(filtered, key=lambda v: counts.get(v, 0), reverse=True)
    else:
        ordered = sorted(unique_values)

    if MAX_SEGMENT_VALUES_PER_DIM:
        ordered = ordered[:MAX_SEGMENT_VALUES_PER_DIM]

    if not ordered:
        continue

    available_segments_by_dim[dim] = ordered
    for v in ordered:
        available_segments.append({'segment_dim': dim, 'segment_value': v})

print(f'Discovered segment dimensions: {list(available_segments_by_dim.keys())}')
print(f'Available segments to render: {len(available_segments)}')
print(f'Sample available segments: {available_segments[:5]}')
print('=' * 60)


SEGMENT DISCOVERY
Segment discovery rules:
  - dimensions considered: ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
  - max values per dimension: 10
  - min rows per segment (raw_sample only): 100
  - null/blank values are skipped
Discovered segment dimensions: ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
Available segments to render: 13
Sample available segments: [{'segment_dim': 'gen', 'segment_value': 'prior_gen'}, {'segment_dim': 'gen', 'segment_value': 'current_gen'}, {'segment_dim': 'region_rollup', 'segment_value': 'nam'}, {'segment_dim': 'region_rollup', 'segment_value': 'eu'}, {'segment_dim': 'customer_segment', 'segment_value': 'bigbird'}]


In [9]:
# ========================================
# SEGMENT NORMALIZATION VALIDATION
# ========================================
print('=' * 60)
print('SEGMENT NORMALIZATION VALIDATION')
print('=' * 60)

norm_check_counts = {}
for test_val in ["Bigbird", "Internal", "bigbird", "internal"]:
    cs = filter_by_segment(corr_summary_pdf, "customer_segment", test_val)
    cot = filter_by_segment(corr_over_time_pdf, "customer_segment", test_val)
    raw = filter_by_segment(raw_sample_pdf, "customer_segment", test_val)

    norm_check_counts[test_val] = {
        'raw_sample': len(raw),
        'corr_summary': len(cs),
        'corr_over_time': len(cot)
    }

    print(f"\nValue={test_val}")
    print(' raw_sample rows     :', len(raw))
    print(' corr_summary rows   :', len(cs))
    print(' corr_over_time rows :', len(cot))

print('\nSample normalized segments:')
for dim in ["customer_segment", "product_segment"]:
    if dim in available_segments_by_dim:
        sample = available_segments_by_dim[dim][:10]
        print(f'  {dim}: {sample}')
    else:
        print(f'  {dim}: (not available)')

corr_sanity_counts = {}
for test_val in ["bigbird", "internal"]:
    cs = filter_by_segment(corr_summary_pdf, "customer_segment", test_val)
    cot = filter_by_segment(corr_over_time_pdf, "customer_segment", test_val)
    corr_sanity_counts[test_val] = {
        'corr_summary': len(cs),
        'corr_over_time': len(cot)
    }
    print(f'\nCorrelation sanity check for customer_segment={test_val}:')
    print(' corr_summary rows   :', len(cs))
    print(' corr_over_time rows :', len(cot))

print('=' * 60)


SEGMENT NORMALIZATION VALIDATION

Value=Bigbird
 raw_sample rows     : 141697
 corr_summary rows   : 630
 corr_over_time rows : 287910

Value=Internal
 raw_sample rows     : 110140
 corr_summary rows   : 630
 corr_over_time rows : 287910

Value=bigbird
 raw_sample rows     : 141697
 corr_summary rows   : 630
 corr_over_time rows : 287910

Value=internal
 raw_sample rows     : 110140
 corr_summary rows   : 630
 corr_over_time rows : 287910

Sample normalized segments:
  customer_segment: ['bigbird', 'internal', 'external-other', 'ai lab']
  product_segment: ['hgx', 'mgx', 'pcie']

Correlation sanity check for customer_segment=bigbird:
 corr_summary rows   : 630
 corr_over_time rows : 287910

Correlation sanity check for customer_segment=internal:
 corr_summary rows   : 630
 corr_over_time rows : 287910


In [10]:
# ========================================
# RENDER GLOBAL VIEWS (CANONICAL)
# ========================================
if RENDER_GLOBAL_VIEWS:
    print('=' * 60)
    print('RENDERING GLOBAL VIEWS')
    print('=' * 60)

    core_pairs = list(combinations(CORE_AVAILABLE, 2))
    selected_core_pairs = core_pairs[:3]

    print('Rendering core density heatmaps...')
    render_core_density_heatmaps(density_bins_pdf)

    print('Rendering monthly density drift...')
    render_monthly_density_drift(density_over_time_pdf, selected_core_pairs, MONTHS)

    print('Rendering distribution charts...')
    render_distribution_charts(raw_sample_pdf, ALL_RAW_METRICS, chart_type='histogram')
    render_distribution_charts(raw_sample_pdf, ALL_RAW_METRICS, chart_type='boxplot')
    render_monthly_distribution_drift(raw_sample_pdf, BOXPLOT_METRICS, MONTHS)

    print('Rendering correlation heatmaps...')
    for corr_group in CORR_GROUPS_TO_RENDER:
        render_correlation_heatmap(corr_summary_pdf, corr_group)

    print('Rendering monthly correlation drift...')
    for corr_group in CORR_GROUPS_TO_RENDER:
        render_monthly_correlation_drift(corr_over_time_pdf, corr_group, MONTHS)

    print('Rendering pairwise scatter plots...')
    scatter_pairs = get_scatter_pairs(
        corr_summary_pdf,
        corr_group='p50',
        top_n=8
        )
    render_pairwise_scatter(raw_sample_pdf, scatter_pairs)

    print('Rendering monthly scatter drift...')
    render_monthly_scatter_drift(raw_sample_pdf, scatter_pairs, MONTHS)

    print('Rendering IQR-scaled drift heatmap...')
    render_iqr_scaled_drift_heatmap(raw_sample_pdf, ALL_RAW_METRICS, MONTHS)

    print('Global views complete.')
    print('=' * 60)


RENDERING GLOBAL VIEWS
Rendering core density heatmaps...
Rendering monthly density drift...
Rendering distribution charts...
Rendering correlation heatmaps...
Rendering monthly correlation drift...
Rendering pairwise scatter plots...
Rendering monthly scatter drift...
Rendering IQR-scaled drift heatmap...

Sample IQR-scaled drift calculations:
  tensor_util: baseline_median=0.036, baseline_iqr=0.252, drift_score@2026-05=-0.005
  gpu_util: baseline_median=0.714, baseline_iqr=0.990, drift_score@2026-05=-0.085
  tensor_tflops: baseline_median=533.271, baseline_iqr=3584.212, drift_score@2026-05=0.013
Global views complete.


In [11]:
# ========================================
# RENDER SEGMENTED VIEWS (CANONICAL)
# ========================================
if RENDER_SEGMENTED_VIEWS:
    print('=' * 60)
    print('RENDERING SEGMENTED VIEWS')
    print('=' * 60)

    chart_family_policy = {
        'static_density_heatmaps': 'segmented-enabled' if {'segment_dim', 'segment_value'}.issubset(density_bins_pdf.columns) else 'global-only (no segmented density bins)',
        'monthly_density_drift': 'segmented-enabled' if {'segment_dim', 'segment_value'}.issubset(density_over_time_pdf.columns) else 'global-only (no segmented density over time)',
        'histograms': 'segmented-enabled' if any(dim in raw_sample_pdf.columns for dim in SEGMENT_DIMS_TO_RENDER) else 'global-only (no segment columns in raw_sample)',
        'boxplots': 'segmented-enabled' if any(dim in raw_sample_pdf.columns for dim in SEGMENT_DIMS_TO_RENDER) else 'global-only (no segment columns in raw_sample)',
        'static_correlation_heatmaps': 'segmented-enabled' if {'segment_dim', 'segment_value'}.issubset(corr_summary_pdf.columns) else 'global-only (no segmented corr_summary)',
        'monthly_correlation_drift': 'segmented-enabled' if {'segment_dim', 'segment_value'}.issubset(corr_over_time_pdf.columns) else 'global-only (no segmented corr_over_time)',
        'pairwise_scatter': 'segmented-enabled' if any(dim in raw_sample_pdf.columns for dim in SEGMENT_DIMS_TO_RENDER) else 'global-only (no segment columns in raw_sample)',
        'monthly_scatter_drift': 'segmented-enabled' if any(dim in raw_sample_pdf.columns for dim in SEGMENT_DIMS_TO_RENDER) else 'global-only (no segment columns in raw_sample)',
        'iqr_scaled_drift_heatmap': 'segmented-enabled' if any(dim in raw_sample_pdf.columns for dim in SEGMENT_DIMS_TO_RENDER) else 'global-only (no segment columns in raw_sample)',
    }

    print('Chart family segmentation status:')
    for k, v in chart_family_policy.items():
        print(f'  - {k}: {v}')

    if not available_segments:
        print('No available segments to render.')
    else:
        core_pairs = list(combinations(CORE_AVAILABLE, 2))
        selected_core_pairs = core_pairs[:2]

        for seg in available_segments:
            seg_dim = seg['segment_dim']
            seg_val = seg['segment_value']
            segment_context = get_segment_context_string(seg_dim, seg_val)
            print(f'\nRendering segment: {segment_context}')

            if chart_family_policy['static_density_heatmaps'].startswith('segmented-enabled'):
                render_core_density_heatmaps(density_bins_pdf, seg_dim, seg_val)

            if chart_family_policy['monthly_density_drift'].startswith('segmented-enabled'):
                render_monthly_density_drift(density_over_time_pdf, selected_core_pairs, MONTHS, seg_dim, seg_val)

            if chart_family_policy['histograms'].startswith('segmented-enabled'):
                render_distribution_charts(raw_sample_pdf, ALL_RAW_METRICS, seg_dim, seg_val, chart_type='histogram')

            if chart_family_policy['boxplots'].startswith('segmented-enabled'):
                render_distribution_charts(raw_sample_pdf, ALL_RAW_METRICS, seg_dim, seg_val, chart_type='boxplot')
                render_monthly_distribution_drift(raw_sample_pdf, BOXPLOT_METRICS, MONTHS, seg_dim, seg_val)

            if chart_family_policy['static_correlation_heatmaps'].startswith('segmented-enabled'):
                for corr_group in CORR_GROUPS_TO_RENDER:
                    render_correlation_heatmap(corr_summary_pdf, corr_group, seg_dim, seg_val)

            if chart_family_policy['monthly_correlation_drift'].startswith('segmented-enabled'):
                for corr_group in CORR_GROUPS_TO_RENDER:
                    render_monthly_correlation_drift(corr_over_time_pdf, corr_group, MONTHS, seg_dim, seg_val)

            if chart_family_policy['pairwise_scatter'].startswith('segmented-enabled'):
                scatter_pairs = get_scatter_pairs(corr_summary_pdf, corr_group='p50', segment_dim=seg_dim, segment_value=seg_val)
                render_pairwise_scatter(raw_sample_pdf, scatter_pairs, seg_dim, seg_val)

            if chart_family_policy['monthly_scatter_drift'].startswith('segmented-enabled'):
                scatter_pairs = get_scatter_pairs(corr_summary_pdf, corr_group='p50', segment_dim=seg_dim, segment_value=seg_val)
                render_monthly_scatter_drift(raw_sample_pdf, scatter_pairs, MONTHS, seg_dim, seg_val)

            if chart_family_policy['iqr_scaled_drift_heatmap'].startswith('segmented-enabled'):
                render_iqr_scaled_drift_heatmap(raw_sample_pdf, ALL_RAW_METRICS, MONTHS, seg_dim, seg_val)

    print('\nSegmented views complete.')
    print('=' * 60)


RENDERING SEGMENTED VIEWS
Chart family segmentation status:
  - static_density_heatmaps: segmented-enabled
  - monthly_density_drift: segmented-enabled
  - histograms: segmented-enabled
  - boxplots: segmented-enabled
  - static_correlation_heatmaps: segmented-enabled
  - monthly_correlation_drift: segmented-enabled
  - pairwise_scatter: segmented-enabled
  - monthly_scatter_drift: segmented-enabled
  - iqr_scaled_drift_heatmap: segmented-enabled

Rendering segment: gen=prior_gen

Rendering segment: gen=current_gen

Rendering segment: region_rollup=nam

Rendering segment: region_rollup=eu


/home/coreweave/.local/lib/python3.11/site-packages/seaborn/matrix.py:202: RuntimeWarning: All-NaN slice encountered
  vmin = np.nanmin(calc_data)
/home/coreweave/.local/lib/python3.11/site-packages/seaborn/matrix.py:207: RuntimeWarning: All-NaN slice encountered
  vmax = np.nanmax(calc_data)



Rendering segment: customer_segment=bigbird

Rendering segment: customer_segment=internal

Rendering segment: customer_segment=external-other

Rendering segment: customer_segment=ai_lab

Rendering segment: product_segment=hgx

Rendering segment: product_segment=mgx

Rendering segment: product_segment=pcie

Rendering segment: is_training=unknown
  No correlation data for p50 in is_training=unknown
  No correlation data for p95 in is_training=unknown
  Skipping monthly correlation drift for is_training=unknown: no segmented data
  Skipping monthly correlation drift for is_training=unknown: no segmented data

Rendering segment: is_training=slurm

Segmented views complete.


In [12]:
# ========================================
# BUILD HTML REPORT + PNG ZIP (CANONICAL)
# ========================================
summary_lines = [
    f'Resolved export root: {resolved_root}',
    f'Created: {datetime.utcnow().isoformat()} UTC',
    f'Trailing month window: {len(MONTHS)} months ({MONTHS[0] if MONTHS else "N/A"} to {MONTHS[-1] if MONTHS else "N/A"})',
    f'Total months available: {len(ALL_MONTHS)} months',
    f'Raw metrics visualized: {len(ALL_RAW_METRICS)}',
    f'Metrics excluded from boxplots: {len(BOXPLOT_EXCLUDE_METRICS)} ({", ".join(BOXPLOT_EXCLUDE_METRICS)})',
    f'Core metrics: {len(CORE_AVAILABLE)}',
    f'Correlation groups rendered: {", ".join(CORR_GROUPS_TO_RENDER)}',
    f'Correlation metrics: {len(CORR_METRICS)}',
    f'Segment dimensions available: {list(available_segments_by_dim.keys()) if available_segments_by_dim else "None"}',
    f'Total segment values rendered: {len(available_segments)}',
    f'Drift heatmap: IQR-scaled relative to baseline month {MONTHS[0] if MONTHS else "N/A"}',
]

add_table('Loaded dataset row counts', pd.DataFrame([
    {'dataset': 'raw_sample', 'rows': len(raw_sample_pdf)},
    {'dataset': 'corr_summary', 'rows': len(corr_summary_pdf)},
    {'dataset': 'corr_over_time', 'rows': len(corr_over_time_pdf)},
    {'dataset': 'density_bins', 'rows': len(density_bins_pdf)},
    {'dataset': 'density_over_time', 'rows': len(density_over_time_pdf)},
    {'dataset': 'bin_metadata', 'rows': len(bin_metadata_pdf)},
    {'dataset': 'trend_summary', 'rows': len(trend_summary_pdf)},
]))

if available_segments_by_dim:
    segment_summary = []
    for dim, values in available_segments_by_dim.items():
        segment_summary.append({'dimension': dim, 'values_count': len(values), 'sample_values': str(values[:3])})
    add_table('Segment dimensions summary', pd.DataFrame(segment_summary))

print('=' * 60)
print('ORDERING VALIDATION (PRE-REPORT)')
print('=' * 60)
print(f'Figures ready: {len(FIGURES)}')
print(f'Report tables ready: {len(REPORT_TABLES)}')
print(f'Report segments ready: {len(REPORT_SEGMENTS)}')
print('=' * 60)

# Write HTML report(s)
if SPLIT_HTML_REPORTS:
    write_split_html_reports(FIGURES, REPORT_TABLES, HTML_REPORT_DIR, summary_lines, REPORT_SEGMENTS)
else:
    write_html_report(FIGURES, REPORT_TABLES, HTML_REPORT_PATH, summary_lines, REPORT_SEGMENTS)
zip_png_pack(FIGURES, PNG_ZIP_PATH)

print('=' * 60)
print('VISUAL OUTPUTS READY')
print('=' * 60)
if SPLIT_HTML_REPORTS:
    print(f'HTML reports: {HTML_REPORT_DIR}')
    print(f'  Main index: {HTML_REPORT_DIR}/index.html')
else:
    print(f'HTML report: {HTML_REPORT_PATH}')
print(f'PNG zip    : {PNG_ZIP_PATH}')
print(f'Total figures generated: {len(FIGURES)}')
print(f'  - Global views: {len(REPORT_SEGMENTS.get("global", []))}')
for segment_context in REPORT_SEGMENTS:
    if segment_context != 'global':
        print(f'  - {segment_context}: {len(REPORT_SEGMENTS[segment_context])}')


ORDERING VALIDATION (PRE-REPORT)
Figures ready: 263
Report tables ready: 2
Report segments ready: 14
Created index: /home/coreweave/dcgm_eda_viz_outputs/html_reports/index.html
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/global.html (21 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/gen=prior_gen.html (19 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/gen=current_gen.html (19 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/region_rollup=nam.html (19 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/region_rollup=eu.html (19 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/customer_segment=bigbird.html (19 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_outputs/html_reports/customer_segment=internal.html (19 figures)
Created segment report: /home/coreweave/dcgm_eda_viz_out

In [13]:
# ========================================
# FINAL VALIDATION + MAINTAINER SUMMARY
# ========================================
print('=' * 60)
print('VALIDATION SUMMARY')
print('=' * 60)

print('✓ Configuration validated:')
print(f'  - MAX_MONTHS set to {MAX_MONTHS}')
print(f'  - CORR_GROUPS_TO_RENDER: {CORR_GROUPS_TO_RENDER}')
print(f'  - BOXPLOT_EXCLUDE_METRICS: {BOXPLOT_EXCLUDE_METRICS}')
print(f'  - Boxplot metrics count: {len(BOXPLOT_METRICS)} (after exclusions)')

print('\n✓ Segmentation discovery:')
print(f'  - Discovered dimensions: {list(available_segments_by_dim.keys()) if available_segments_by_dim else "None"}')
print(f'  - Available segments: {len(available_segments)}')
print(f'  - Sample segments: {available_segments[:5]}')

print('\n✓ Chart family coverage:')
print('  - Global views: {}'.format('Enabled' if RENDER_GLOBAL_VIEWS else 'Disabled'))
print('  - Segmented views: {}'.format('Enabled' if RENDER_SEGMENTED_VIEWS else 'Disabled'))
print('  - Chart families evaluated for segmentation:')
print('    • static density heatmaps')
print('    • monthly density drift views')
print('    • histograms')
print('    • boxplots')
print('    • static correlation heatmaps (p50/p95)')
print('    • monthly correlation drift (p50/p95)')
print('    • pairwise scatter plots')
print('    • monthly scatter drift')
print('    • IQR-scaled drift heatmap')

print('\n✓ Ordering validation:')
print(f'  - Figures created: {len(FIGURES)}')
print(f'  - Report tables created: {len(REPORT_TABLES)}')
print(f'  - Report segments created: {len(REPORT_SEGMENTS)}')

print('\n✓ Drift validation:')
print('  - Drift heatmap uses (current - baseline_median) / baseline_IQR')
print(f'  - Baseline month: {MONTHS[0] if MONTHS else "N/A"}')

print('\n✓ Duplicate-path validation:')
print('  - One report-generation path (canonical)')
print('  - One drift-heatmap path (IQR-scaled)')
print('  - One segmented render loop')

print('\nMaintainer summary:')
print('  • available_segments defined in "SEGMENT DISCOVERY (CANONICAL)" section')
print('  • Removed legacy duplicate report-generation and drift-heatmap sections')
print('  • Segmented chart families now rendered through the canonical segmented loop')
print('  • Only one final report build path remains')
print('  • IQR-scaled drift is the only active drift implementation')
print('=' * 60)
print('\nSegment normalization summary:')
print('  - Root cause: segment values differed only by casing between raw_sample and parquet-derived segment_value fields.')
print('  - Normalization added: normalize_segment_value + STRING_SEGMENT_DIMS in segmentation helpers.')
print('  - Normalized comparisons applied in discover_segment_values, segment discovery, and filter_by_segment for raw and segmented datasets.')

if 'norm_check_counts' in globals():
    print('  - Bigbird/Internal validation counts:')
    for key in ['Bigbird', 'bigbird', 'Internal', 'internal']:
        if key in norm_check_counts:
            counts = norm_check_counts[key]
            print(f"    * {key}: raw_sample={counts['raw_sample']}, corr_summary={counts['corr_summary']}, corr_over_time={counts['corr_over_time']}")

if 'corr_sanity_counts' in globals():
    for key, counts in corr_sanity_counts.items():
        print(f"  - Correlation sanity ({key}): corr_summary={counts['corr_summary']}, corr_over_time={counts['corr_over_time']}")



VALIDATION SUMMARY
✓ Configuration validated:
  - MAX_MONTHS set to 15
  - CORR_GROUPS_TO_RENDER: ['p50', 'p95']
  - BOXPLOT_EXCLUDE_METRICS: ['vram_usage', 'redfish_power', 'chip_power', 'sm_clock', 'tensor_tflops', 'tensor_tflops_sm']
  - Boxplot metrics count: 12 (after exclusions)

✓ Segmentation discovery:
  - Discovered dimensions: ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
  - Available segments: 13
  - Sample segments: [{'segment_dim': 'gen', 'segment_value': 'prior_gen'}, {'segment_dim': 'gen', 'segment_value': 'current_gen'}, {'segment_dim': 'region_rollup', 'segment_value': 'nam'}, {'segment_dim': 'region_rollup', 'segment_value': 'eu'}, {'segment_dim': 'customer_segment', 'segment_value': 'bigbird'}]

✓ Chart family coverage:
  - Global views: Enabled
  - Segmented views: Enabled
  - Chart families evaluated for segmentation:
    • static density heatmaps
    • monthly density drift views
    • histograms
    • boxplots
    • static corre

In [14]:
# for test_val in ["Bigbird", "bigbird", "Internal", "internal"]:
#     raw = filter_by_segment(raw_sample_pdf, "customer_segment", test_val)
#     cs = filter_by_segment(corr_summary_pdf, "customer_segment", test_val)
#     cot = filter_by_segment(corr_over_time_pdf, "customer_segment", test_val)

#     print(f"\nValue={test_val}")
#     print(" raw_sample rows     :", len(raw))
#     print(" corr_summary rows   :", len(cs))
#     print(" corr_over_time rows :", len(cot))
